# Lecture 2 · From training data to a practical language model (Fable edition)

Companion notebook, generated from the code on the lecture pages (https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-02-fable/). Run the cells in order; each one corresponds to a numbered step on the pages. Steps marked 💳 need `LLM_API_KEY` (and optionally `LLM_BASE_URL`, `LLM_MODEL`) in the environment and are skipped otherwise.

## Step 1 — Score a section of the notes in English, in Ukrainian and shuffled

[1. Measure what training optimises and what it costs](https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-02-fable/01-measure-what-training-optimises/)

In [ ]:
import json, math, pathlib, random, re, time, urllib.request
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

BASE, INSTRUCT = ("HuggingFaceTB/SmolLM2-135M",
                  "HuggingFaceTB/SmolLM2-135M-Instruct")
tok = AutoTokenizer.from_pretrained(BASE)
# float32: fast on any CPU
base = AutoModelForCausalLM.from_pretrained(BASE, dtype=torch.float32).eval()

CORPUS = ("https://aiml-study-guides.onrender.com/corpus/generative-agentic-"
          "ai/lectures-01-06.{}.jsonl")
notes = {}
for lang in ("en", "uk"):
    path = pathlib.Path(f"lectures-01-06.{lang}.jsonl")
    if not path.exists():
        urllib.request.urlretrieve(CORPUS.format(lang), path)
    notes[lang] = {(r["lecture"], r["section"]): r["text"]
                   for r in map(json.loads, path.open(encoding="utf-8"))
                   if r["section"]}

@torch.no_grad()
def loss_of(text, model=base, max_tokens=512):
    """Take text; return mean loss (nats), token count and the text scored."""
    ids = tok(text, return_tensors="pt")["input_ids"][:, :max_tokens]
    return model(ids, labels=ids).loss.item(), ids.shape[1], tok.decode(ids[0])

key = (2, "1.1")  # the same section of the notes in both languages
random.seed(0)
words = notes["en"][key].split()
samples = {
    "English notes": notes["en"][key],
    "the same notes in Ukrainian": notes["uk"][key],
    "English notes, words shuffled": " ".join(random.sample(words, len(words)))
}
for name, text in samples.items():
    loss, n, covered = loss_of(text)
    # total bits / UTF-8 bytes
    bits_per_byte = loss * (n - 1) / math.log(2) / len(covered.encode("utf-8"))
    print(f"{name:<30} {n:>3} tokens  loss {loss:5.2f}  perplexity "
          f"{math.exp(loss):6.1f}  bits per byte {bits_per_byte:4.2f}")

## Step 2 — Compute the training bill 6ND for five models

[1. Measure what training optimises and what it costs](https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-02-fable/01-measure-what-training-optimises/)

In [ ]:
# SmolLM2-135M's parameters, counted
N = sum(p.numel() for p in base.parameters())
MODELS = {  # parameters, training tokens (sources on the page)
    "SmolLM2-135M": (N, 2e12),
    "SmolLM2-1.7B": (1.7e9, 11e12),
    "Llama 3 8B": (8e9, 15e12),
    "Chinchilla": (70e9, 1.4e12),
    "GPT-3": (175e9, 300e9),
}
GPU = 400e12  # illustrative: 400 TFLOP/s sustained by one modern GPU
print(f"SmolLM2-135M has {N:,} parameters")
print(
    f"{'model':<14}{'parameters':>12}{'tokens':>10}"
    f"{'FLOPs = 6ND':>14}{'GPU-days':>11}"
)
for name, (n, d) in MODELS.items():
    flops = 6 * n * d
    print(
        f"{name:<14}{n:>12.3g}{d:>10.2g}{flops:>14.2e}"
        f"{flops / GPU / 86400:>11,.0f}"
    )

## Step 3 — Compare each model with the Chinchilla balance for its own budget

[1. Measure what training optimises and what it costs](https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-02-fable/01-measure-what-training-optimises/)

In [ ]:
# Chinchilla's rule of thumb: for a fixed compute budget C = 6ND, loss is
# lowest near D ≈ 20 N.
for name, (n, d) in MODELS.items():
    c = 6 * n * d
    n_opt = (c / (6 * 20)) ** 0.5  # solve C = 6 · N · (20 N) for N
    print(f"{name:<14} {d / n:>7,.0f} tokens per parameter   compute-optimal "
          f"for its budget: "
          f"{n_opt:.2g} parameters on {20 * n_opt:.2g} tokens")

## Step 4 — Filter the notes, find near-duplicates and personal data

[2. Data decide what the model knows](https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-02-fable/02-data-decide-what-the-model-knows/)

In [ ]:
import hashlib
docs = dict(notes["en"])  # 1 document = 1 section of the English notes
# a near-copy, made on purpose
docs[("planted", "copy")] = docs[(2, "1.1")].replace("model", "network", 3)
docs[("planted", "pii")] = (
    "Contact the tutor at jane.doe@example.org or +49 170 1234567 "
    "for the answers.")

# rules in the spirit of the Gopher filters (source on the page)
def quality_flags(text):
    """Take a text; return the list of quality rules it breaks."""
    w = text.split()
    flags = []
    if len(w) < 50: flags.append("fewer than 50 words")
    if not 3 <= sum(map(len, w)) / max(len(w), 1) <= 10:
        flags.append("odd mean word length")
    if sum(t in "#{}<>|" for t in text) / max(len(w), 1) > 0.1:
        flags.append("many symbols")
    if re.search(r"[\w.+-]+@[\w-]+\.\w+|\+?\d[\d ]{8,}\d", text):
        flags.append("personal data (email/phone)")
    return flags

def minhash(text, k=64, n=5):  # 64 hash minimums over 5-word shingles
    """Take a text; return its signature: k minimum hash values."""
    w = text.lower().split()
    shingles = {" ".join(w[i:i + n]) for i in range(len(w) - n + 1)} or {text}
    return [
        min(
            int(hashlib.md5(f"{s}|{j}".encode()).hexdigest()[:12], 16)
            for s in shingles) for j in range(k)
    ]

flagged = {
    k: f
    for k, f in ((k, quality_flags(t)) for k, t in docs.items()) if f
}
print(f"{len(docs)} documents; flagged by the quality rules: {len(flagged)}")
for k, f in list(flagged.items())[:4]:
    print("  ", k, f)
sig = {k: minhash(t) for k, t in docs.items()}
keys = list(sig)
pairs = [(a, b, sum(x == y for x, y in zip(sig[a], sig[b])) / 64)
         for i, a in enumerate(keys) for b in keys[i + 1:]]
near = sorted([p for p in pairs if p[2] >= 0.5], key=lambda p: -p[2])
print("near-duplicate pairs (estimated Jaccard ≥ 0.5):",
      [(a, b, round(s, 2)) for a, b, s in near])

## Step 5 — Ask the base and the instruct model the same three things

[3. From text predictor to assistant](https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-02-fable/03-from-text-predictor-to-assistant/)

In [ ]:
# same vocabulary + a chat template
chat_tok = AutoTokenizer.from_pretrained(INSTRUCT)
# same network, after SFT + DPO
inst = AutoModelForCausalLM.from_pretrained(INSTRUCT,
                                            dtype=torch.float32).eval()

prompts = ["What is the capital of France?",
           "List three tips for learning a new language.",
           # "Write a 6-line poem in Ukrainian about autumn."
           "Напиши вірш з 6 рядків українською про осінь."]

@torch.no_grad()
def reply(model, text, chat, n=48):
    """Take a model and a prompt, chat-formatted or not; return its reply."""
    ids = (chat_tok.apply_chat_template(
        [{"role": "user", "content": text}], add_generation_prompt=True,
        return_tensors="pt", return_dict=True)["input_ids"] if chat else tok(
            text, return_tensors="pt")["input_ids"])
    out = model.generate(ids, max_new_tokens=n, do_sample=False,
                         pad_token_id=tok.eos_token_id)
    return tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True).strip()

for p in prompts:
    print("PROMPT:", p)
    print("  base     →",
          reply(base, p, chat=False).replace("\n", " ⏎ ")[:220])
    print("  instruct →", reply(inst, p, chat=True).replace("\n", " ⏎ ")[:220])

## Step 6 — Measure what post-training changed on three preference pairs

[3. From text predictor to assistant](https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-02-fable/03-from-text-predictor-to-assistant/)

In [ ]:
pairs = [  # (prompt, chosen, rejected): factual, format, honest
    ("What is the capital of France?", "The capital of France is Paris.",
     "The capital of France is Marseille."),
    ("Give three short tips for sleeping well, as bullet points.",
     "- Go to bed at the same time.\n- Keep the room dark.\n- Avoid "
     "screens before bed.",
     "Sleeping well is important, and there are many things you can do to "
     "improve it, such as having a routine."),
    ("What does the Python keyword `unless` do?",
     "Python has no `unless` keyword; write `if not condition:` instead.",
     "`unless` runs a block of code only when the condition is false."),
]

@torch.no_grad()
def logp(model, prompt, answer):
    """Take a model, prompt and answer; return the answer's summed log-p."""
    ctx = chat_tok.apply_chat_template([{"role": "user", "content": prompt}],
                                       add_generation_prompt=True,
                                       tokenize=False)
    p_ids = chat_tok(ctx, return_tensors="pt")["input_ids"]
    a_ids = chat_tok(answer, return_tensors="pt",
                     add_special_tokens=False)["input_ids"]
    ids = torch.cat([p_ids, a_ids], dim=1)
    logprobs = model(ids).logits[0, :-1].log_softmax(-1)
    targets = ids[0, 1:]
    return logprobs[p_ids.shape[1] - 1:].gather(
        1, targets[p_ids.shape[1] - 1:, None]).sum().item()

beta = 0.1
for prompt, good, bad in pairs:
    r = {}
    for name, y in (("chosen", good), ("rejected", bad)):
        lb, li = logp(base, prompt, y), logp(inst, prompt, y)
        # implicit reward: how much instruct raised this answer
        r[name] = beta * (li - lb)
        print(f"{prompt[:34]:<35} {name:<8} log p base {lb:7.1f}  instruct "
              f"{li:7.1f}  reward {r[name]:+.2f}")
    margin = r["chosen"] - r["rejected"]
    print(
        f"{'':<35} margin {margin:+.2f}  DPO loss "
        f"{-torch.nn.functional.logsigmoid(torch.tensor(margin)).item():.3f}")

## Step 7 — Score eight sampled answers with a verifier and compare them with their group

[3. From text predictor to assistant](https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-02-fable/03-from-text-predictor-to-assistant/)

In [ ]:
task, truth = ("Tom has 12 apples and gives away 5. How many apples does "
               "Tom have now?"), 7

def verify(text):
    """Checkable reward: 1.0 if the text's last number is truth, else 0.0."""
    nums = re.findall(r"-?\d+", text)
    return float(bool(nums) and int(nums[-1]) == truth)

ids = chat_tok.apply_chat_template([{"role": "user", "content": task}],
                                   add_generation_prompt=True,
                                   return_tensors="pt",
                                   return_dict=True)["input_ids"]
torch.manual_seed(0)
with torch.no_grad():
    out = inst.generate(ids.repeat(8, 1), max_new_tokens=96, do_sample=True,
                        temperature=0.8, pad_token_id=tok.eos_token_id)
answers = [
    chat_tok.decode(o[ids.shape[1]:], skip_special_tokens=True).strip()
    for o in out
]
rewards = torch.tensor([verify(a) for a in answers])
# GRPO: compare each answer with its own group
adv = (rewards - rewards.mean()) / (rewards.std() + 1e-6)
for a, r, g in zip(answers, rewards, adv):
    print(
        f"reward {r:.0f}  advantage {g:+.2f}  {a.replace(chr(10), ' ')[-80:]}")
print(f"group mean reward {rewards.mean():.3f}")

## Step 8 — Measure the baseline: memory, speed and perplexity

[4. Make the model cheap enough to run](https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-02-fable/04-make-the-model-cheap-enough-to-run/)

In [ ]:
import io, copy
# Lecture 2 notes, English
eval_text = "\n\n".join(t for (lec, sec), t in notes["en"].items() if lec == 2)
# one fixed 1,024-token test text
eval_ids = tok(eval_text, return_tensors="pt", truncation=True,
               max_length=1024)["input_ids"]

@torch.no_grad()
def perplexity(model):
    """Take a model; return its perplexity on the fixed test text."""
    return math.exp(model(eval_ids, labels=eval_ids).loss.item())

@torch.no_grad()
def tokens_per_second(model, n=64):
    """Time n greedy new tokens after a warm-up; return tokens per second."""
    x = tok("The course notes say that", return_tensors="pt")["input_ids"]
    # warm-up
    model.generate(x, max_new_tokens=4, do_sample=False,
                   pad_token_id=tok.eos_token_id)
    t = time.perf_counter()
    model.generate(x, max_new_tokens=n, min_new_tokens=n, do_sample=False,
                   pad_token_id=tok.eos_token_id)
    return n / (time.perf_counter() - t)

def saved_mb(model):
    """Save the model's state dict in memory; return its size in MB."""
    buf = io.BytesIO(); torch.save(model.state_dict(), buf)
    return buf.tell() / 1e6

n_params = sum(p.numel() for p in inst.parameters())
print(
    f"parameters {n_params:,}: float32 {n_params * 4 / 1e6:.0f} MB, bfloat16 "
    f"{n_params * 2 / 1e6:.0f} MB, 4-bit {n_params / 2 / 1e6:.0f} MB")
print(f"saved state dict (float32) {saved_mb(inst):.0f} MB; CPU threads "
      f"{torch.get_num_threads()}")
baseline = {
    "size": saved_mb(inst), "speed": tokens_per_second(inst),
    "ppl": perplexity(inst)
}
print(f"instruct, float32: {baseline['speed']:.1f} tokens/s, perplexity "
      f"{baseline['ppl']:.2f} on {eval_ids.shape[1]} tokens of notes")

## Step 9 — Quantize one matrix by hand, then the whole model with torchao

[4. Make the model cheap enough to run](https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-02-fable/04-make-the-model-cheap-enough-to-run/)

In [ ]:
import warnings
warnings.filterwarnings("ignore")
from torchao.quantization import (quantize_, Int8WeightOnlyConfig,
                                  Int8DynamicActivationInt8WeightConfig,
                                  IntxWeightOnlyConfig)
from torchao.quantization.granularity import PerGroup

# symmetric round-to-nearest, one scale per group of weights
def fake_quant(w, bits, group):
    """Round w to bits, a scale per group; return the values read back."""
    g = w.reshape(-1, group)
    scale = g.abs().amax(1, keepdim=True) / (2 ** (bits - 1) - 1)
    return ((g / scale).round().clamp(-(2**(bits - 1)), 2**(bits - 1) - 1) *
            scale).reshape(w.shape)

# one real matrix, 1536 × 576
w = inst.model.layers[0].mlp.up_proj.weight.detach()
for bits, group in ((8, 576), (4, 576), (4, 32)):
    err = (fake_quant(w, bits, group) - w).norm() / w.norm()
    print(f"one matrix, {bits}-bit, a scale per {group:>3} weights: "
          f"relative error {err:.3f}")

# embeddings stay float32
only_linear = lambda m, name: isinstance(m, torch.nn.Linear
                                         ) and "lm_head" not in name
for name, cfg in (("int8 weights", Int8WeightOnlyConfig()),
                  ("int8 weights + activations",
                   Int8DynamicActivationInt8WeightConfig()),
                  ("int4 weights, groups of 32",
                   IntxWeightOnlyConfig(weight_dtype=torch.int4,
                                        granularity=PerGroup(32)))):
    q = copy.deepcopy(inst); quantize_(q, cfg, filter_fn=only_linear)
    print(
        f"{name:<27} saved {saved_mb(q):4.0f} MB  "
        f"{tokens_per_second(q):5.1f} tokens/s  perplexity "
        f"{perplexity(q):.2f}", flush=True)

## Step 10 — Compare the teacher’s soft targets with the one-hot label

[4. Make the model cheap enough to run](https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-02-fable/04-make-the-model-cheap-enough-to-run/)

In [ ]:
teacher = AutoModelForCausalLM.from_pretrained(
    "HuggingFaceTB/SmolLM2-360M-Instruct", dtype=torch.float32).eval()
print(f"teacher perplexity {perplexity(teacher):.2f} vs student "
      f"{baseline['ppl']:.2f} on the same notes")

with torch.no_grad():
    # same tokenizer, so the vocabularies line up
    t_logp = teacher(eval_ids).logits[0, :-1].log_softmax(-1)
    s_logp = inst(eval_ids).logits[0, :-1].log_softmax(-1)
targets = eval_ids[0, 1:]
entropy = -(t_logp.exp() * t_logp).sum(-1) / math.log(2)  # bits per position
kl = (t_logp.exp() * (t_logp - s_logp)).sum(-1)  # KL(teacher ‖ student), nats
# the usual loss: one correct token
hard = -s_logp.gather(1, targets[:, None]).squeeze(1)
print(f"teacher's targets carry {entropy.mean():.2f} bits of uncertainty per "
      f"token on average")
print(f"student: loss on the one correct token {hard.mean():.2f} nats; KL to "
      f"the teacher's distribution {kl.mean():.2f} nats")

i = int((entropy > 2).nonzero()[5])  # a position where the teacher hesitates
print("context:", repr(tok.decode(eval_ids[0, max(0, i - 8):i + 1])),
      "→ correct:", repr(tok.decode(targets[i])))
for p, j in zip(*t_logp[i].exp().topk(5)):
    print(f"   teacher {p:5.1%}  student {s_logp[i, j].exp():5.1%}  "
          f"{tok.decode(j)!r}")

## Step 11 — Zero 10, 30 and 50 % of the MLP weights, smallest first or at random

[4. Make the model cheap enough to run](https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-02-fable/04-make-the-model-cheap-enough-to-run/)

In [ ]:
def pruned(model, share, by="magnitude"):
    """Copy model, zero a share of MLP weights (by size/random); return it."""
    m = copy.deepcopy(model)
    g = torch.Generator().manual_seed(0)
    with torch.no_grad():
        for layer in m.model.layers:
            for lin in (layer.mlp.gate_proj, layer.mlp.up_proj,
                        layer.mlp.down_proj):
                w = lin.weight
                score = w.abs() if by == "magnitude" else torch.rand(
                    w.shape, generator=g)
                k = int(share * w.numel())
                # zero the k lowest-scoring weights
                w[score <= score.flatten().kthvalue(k).values] = 0
    return m

mlp = sum(l.weight.numel() for layer in inst.model.layers
          for l in (layer.mlp.gate_proj, layer.mlp.up_proj,
                    layer.mlp.down_proj))
print(f"MLP weights: {mlp:,} of {n_params:,} parameters "
      f"({mlp / n_params:.0%}); baseline perplexity {baseline['ppl']:.2f}")
for share in (0.1, 0.3, 0.5):
    print(
        f"zero {share:.0%} of MLP weights: smallest first → perplexity "
        f"{perplexity(pruned(inst, share)):7.2f};"
        f" chosen at random → "
        f"{perplexity(pruned(inst, share, by='random')):9.2f}", flush=True)